In [1]:
import os 


from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# load the api keys 
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGCHAIN_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


In [3]:
os.environ["GEMINI_API_KEY"]=GEMINI_API_KEY
os.environ["LANGCHAIN_API_KEY"]=LANGCHAIN_API_KEY
os.environ["GROQ_API_KEY"]=GROQ_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

In [4]:
# Langsmith tracing 

LANGSMITH_TRACING="true"
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_API_KEY=LANGCHAIN_API_KEY
LANGSMITH_PROJECT="pr-slight-childhood-100"
GEMINI_API_KEY=GEMINI_API_KEY

In [5]:
# Checking which model to use from the gemini 
import google.generativeai as genai

d:\GENAI-2\Langgraph\prerequisities\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
models = genai.list_models()
for model in models:
    print(model.name)

models/embedding-gecko-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-04-17
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash
models/gemini-2.5-flash-preview-04-17-thinking
models/gemini-2.5-flash-lite-preview-06-17
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-ex

In [7]:
# importing important langchain libraries 
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma

from langchain.text_splitter import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [8]:
# loading the gemini models 
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

g_embedding = GoogleGenerativeAIEmbeddings(model="models/embedding-004",google_api_key=GEMINI_API_KEY)
llm_model = ChatGoogleGenerativeAI(model="models/gemini-1.5-flash",google_api_key=GEMINI_API_KEY)

**Simple AI Assistance**

In [9]:
while True:
    Question = input("Enter the Question if you have any. If not just enter quit")
    if Question != "quit":
        print(llm_model.invoke(Question).content)
    else:
        print("Good bye take care")
        break

Good bye take care


In [10]:
# But this assistance is unable to track the history 
# introducing an memory into it 

from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.chat_history  import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_core.messages import AIMessage

In [11]:
# storing history here 
store={}

def get_session_history(session_id :str) -> BaseChatMessageHistory:
    if session_id not in store:
        store['session_id'] = InMemoryChatMessageHistory()
    return store['session_id']


In [12]:
config = {"configurable":{"session_id":"firstchat"}}

In [13]:
# model with memory 
model_with_memory = RunnableWithMessageHistory(llm_model,get_session_history)

In [14]:
model_with_memory.invoke(("Hi my nmae is Yogesh"),config=config).content

"Hi Yogesh!  It's nice to meet you."

In [15]:
# asking for memory from the previous session 
model_with_memory.invoke(("above i have given my name to you ?"), config=config).content
# this is not including the history becase it should be incoperated into thee prompt 

'You have not given me your name.  I have no memory of past conversations.  Each interaction with me starts fresh.'

**Building RAG**

In [16]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma 
from langchain import PromptTemplate    
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [17]:
# readign the data 
loader = DirectoryLoader("D:\GENAI-2\Langgraph\prerequisities\data",loader_cls=PyPDFLoader)

In [18]:
docs = loader.load()

In [19]:
# texxt splitter 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,

)

new_docs = text_splitter.split_documents(documents=docs)

doc_string = [doc.page_content for doc in new_docs]
print(doc_string)

['You Only Look Once:', 'Uniﬁed, Real-Time Object Detection', 'Joseph Redmon∗, Santosh Divvala∗†, Ross', 'Ross Girshick¶, Ali Farhadi∗†', 'University of Washington∗, Allen Institute for', 'for AI†, Facebook AI Research¶', 'http://pjreddie.com/yolo/\nAbstract', 'We present YOLO, a new approach to object', 'to object detection.', 'Prior work on object detection repurposes', 'classiﬁers to per-', 'form detection. Instead, we frame object', 'object detection as a re-', 'gression problem to spatially separated bounding', 'bounding boxes and', 'associated class probabilities. A single neural', 'neural network pre-', 'dicts bounding boxes and class probabilities', 'directly from', 'full images in one evaluation. Since the whole', 'the whole detection', 'pipeline is a single network, it can be optimized', 'optimized end-to-end', 'directly on detection performance.', 'Our uniﬁed architecture is extremely fast. Our', 'fast. Our base', 'YOLO model processes images in real-time at 45', 'at 45 fram

In [20]:
# Set the db 

g_embedding = GoogleGenerativeAIEmbeddings(model="models/embedding-001",google_api_key=GEMINI_API_KEY)
db = Chroma.from_documents(new_docs,g_embedding)
retriever = db.as_retriever(search_kwargs={"k":4})

In [21]:
# design the promt 

template = """
Answer the following question based on the followign context:
{context}
Question: {Question}

"""

prompt = PromptTemplate.from_template(template)

In [22]:
parser = StrOutputParser()

In [23]:
retriever_chain = (
    RunnableParallel({"context":retriever,"Question":RunnablePassthrough()})
    | prompt
    | llm_model
    | parser
)

In [24]:
retriever_chain.invoke("form given document,Tell me what is Yolo ?")

"Based on the provided text snippets, YOLO is described as:\n\n* **Extremely fast:**  It's highlighted for its speed.\n* **Refreshingly simple:** Its design is considered straightforward.\n* **A general-purpose** object detection system: It can detect objects.\n* **Able to detect the size and shape of objects:**  It's capable of determining object dimensions.\n\nThe documents mention YOLO in the context of object detection, emphasizing its speed and simplicity.  A figure (Figure 1, not shown here) is referenced as illustrating its design."

In [25]:
retriever_chain.invoke("Summerize the yolo in 50 words")

'Based on the provided text snippets, YOLO is a method that combines results.  The text mentions comparing YOLO to other methods (Zeiler-).  More information is needed for a complete summary.'

**Tool and Agents**

In [26]:
from langchain_community.tools import WikipediaQueryRun 
from langchain_community.utilities import WikipediaAPIWrapper 

In [27]:
# api wrapper
api_wrapper = WikipediaAPIWrapper(top_k_results=5,doc_content_chars_max=50)


In [28]:
# tool 

tool = WikipediaQueryRun(api_wrapper=api_wrapper)

In [29]:
print(tool.name)

wikipedia


In [30]:
print(tool.description)

A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.


In [31]:
print(tool.args)

{'query': {'description': 'query to look up on wikipedia', 'title': 'Query', 'type': 'string'}}


In [32]:
tool.run({"query":"What is Data Science"})

'Page: Data science\nSummary: Data science is an int'

**YouTube Search tool**

In [33]:
from langchain_community.tools import YouTubeSearchTool

In [34]:
tool1 = YouTubeSearchTool()

In [35]:
tool1.run("Krish naik")

"['https://www.youtube.com/watch?v=JxgmHe2NyeY&pp=ygUKS3Jpc2ggbmFpaw%3D%3D', 'https://www.youtube.com/watch?v=TYEqenKrbaM&pp=ygUKS3Jpc2ggbmFpaw%3D%3D']"

**WebSearch Tool**

In [50]:
from langchain_tavily import TavilySearch
search = TavilySearch(max_results=2)
search_result = search.invoke("What is the current weather condition in SF")
print(search_result)

{'query': 'What is the current weather condition in SF', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in San Francisco', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.775, 'lon': -122.4183, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1752066328, 'localtime': '2025-07-09 06:05'}, 'current': {'last_updated_epoch': 1752066000, 'last_updated': '2025-07-09 06:00', 'temp_c': 14.4, 'temp_f': 57.9, 'is_day': 1, 'condition': {'text': 'Overcast', 'icon': '//cdn.weatherapi.com/weather/64x64/day/122.png', 'code': 1009}, 'wind_mph': 4.3, 'wind_kph': 6.8, 'wind_degree': 281, 'wind_dir': 'WNW', 'pressure_mb': 1020.0, 'pressure_in': 30.13, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 87, 'cloud': 100, 'feelslike_c': 14.4, 'feelslike_f': 57.9, 'windchill_c': 12.2, 'windchill_f': 53.9, 'heatindex_c': 12.9, 'heatindex_f': 55.3, 'dewpoin

In [36]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [37]:
tool2 = TavilySearchResults(tavily_api_key="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")
tool2.run({"query":"What is the current news in Maharashtra politics in India ?"})

C:\Users\yogesh\AppData\Local\Temp\ipykernel_17500\1099840443.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tool2 = TavilySearchResults(tavily_api_key="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")


[{'title': 'Maharashtra politics - The Indian Express',
  'url': 'https://indianexpress.com/about/maharashtra-politics/',
  'content': "follow us on Facebook\nfollow us on Twitter\nfollow us on youtube\nfollow us on Instagram\nThe Indian Express logo\n\n# Maharashtra politics\n\n## MAHARASHTRA POLITICS NEWS\n\nsanjay gaikwad\n\n### Devendra Fadnavis criticises Sanjay Gaikwad, Sena MLA who beat canteen worker; puts onus for action on presiding officers of legislature\n\nJuly 09, 2025 3:51 pm\n\nShiv Sena MLA Sanjay Gaikwad allegedly assaulted an employee in the canteen of the Akashwani legislators' residence over 'stale food' on Tuesday. [...] May 18, 2025 10:20 am\n\n“We would like to retain the alliance in the polls. But in local bodies, you cannot impose decisions from the top,” says state BJP president Chandrashekhar Bawankule\n\ndhananjay munde\n\n### Who is Dhananjay Munde, the Maharashtra minister sacked over sarpanch murder row?\n\nMarch 04, 2025 2:25 pm [...] ### After CM Deven

**cREATING AGENT FROM TEH TOOL**


In [38]:
from langchain import hub
from langchain.agents import AgentExecutor,create_openai_functions_agent


In [39]:
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools.tavily_search import TavilySearchResults

# Model
llm = ChatGoogleGenerativeAI(model="models/gemini-1.5-flash",google_api_key=GEMINI_API_KEY)

# Tool
tools = [TavilySearchResults(tavily_api_key="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")]

# ✅ Gemini-safe prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "{instructions}"),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad", optional=True)  # ✅ optional = avoids empty part error
]).partial(instructions="You are a helpful assistant")

# ✅ Create agent (not openai-functions-agent!)
agent = create_tool_calling_agent(llm, tools, prompt)

# Agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Call
response = agent_executor.invoke({"input": "Who is Bhagat Singh?"})
print(response)




> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'Who is Bhagat Singh?'}`


[{'title': 'Bhagat Singh | Biography, Death, Lahore Conspiracy Case, Martyrs ...', 'url': 'https://www.britannica.com/biography/Bhagat-Singh', 'content': 'Bhagat Singh (born September 27/28, 1907, Banga, Lyallpur, western Punjab, India [now Faisalabad, Pakistan]—died March 23, 1931, Lahore [now in Pakistan]) was an Indian freedom fighter who was honored as a hero of the Indian Independence Movement after his execution by British authorities at the age of 23. He is also known as Shaheed Bhagat Singh (shaheed meaning “martyr”).\n\n## Early life [...] Bhagat Singh was a hero of the early 20th-century Indian Independence Movement. He was a vocal critic of British raj in India and was involved in two high-profile attacks on British authorities—one on a local police chief and the other on the Central Legislative Assembly in Delhi. He was executed for his revolutionary ac

**Creating AGent in the Langchain**

1.Create tool_calling_agent

In [51]:
# connect tavily key 
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

In [54]:
# import langchain_tavily 
from langchain_tavily import TavilySearch
# create tool
search = TavilySearch(max_results=1)
search_result = search.invoke("Who is Priminister of Pakistan ?")
print(search_result) 

{'query': 'Who is Priminister of Pakistan ?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://x.com/pakpmo?lang=en', 'title': "Prime Minister's Office (@PakPMO) / X", 'content': "Prime Minister's Office (@PakPMO) / X Sign up Prime Minister's Office Prime Minister's Office 3.7M Followers Prime Minister's Office’s posts Prime Minister's Office Prime Minister Muhammad Shehbaz Sharif chairs a consultative meeting on preparation of the Federal Budget 2025-2026. Prime Minister's Office Prime Minister Muhammad Shehbaz Sharif addressing the Nation. Prime Minister's Office Prime Minister Muhammad Shehbaz Sharif addresses the National Assembly session. Prime Minister's Office Prime Minister Muhammad Shehbaz Sharif chairs a meeting of the National Security Committee. Prime Minister's Office Ambassador of Kuwait to Pakistan H.E. Nasser Abdulrahman Jasser called on Prime Minister Muhammad Shehbaz Sharif in Islamabad. Prime Minister's Office Ambassador of UAE 

In [57]:
# make tool of it 
tool = [search]

In [58]:
# import prompt 
from langchain import hub
prompt = hub.pull("hwchase17/openai-functions-agent")

In [ ]:
# import the agent 
from langchain.agents import create_tool_calling_agent
agent = create_tool_calling_agent(
    llm=llm,
    tools=tool,
    prompt=prompt
)

# this is a tool_calling_agent

In [62]:
# import agent_executor and create it 
from langchain.agents import AgentExecutor

agent_executor = AgentExecutor(
    agent=agent,
    tools=tool,
    verbose=True
)

In [64]:
result = agent_executor.invoke({"input":"What is Agentic AI ?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search` with `{'search_depth': 'advanced', 'query': 'What is Agentic AI?'}`


{'query': 'What is Agentic AI?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://en.wikipedia.org/wiki/Agentic_AI', 'title': 'Agentic AI - Wikipedia', 'content': 'Agentic AI is a class of artificial intelligence that focuses on autonomous systems that can make decisions and perform tasks without human intervention. The independent systems automatically respond to conditions, to produce process results. The field is closely linked to agentic automation, also known as agent-based process management systems, when applied to process automation. Applications include software development, customer support, cybersecurity and business intelligence. [...] The core concept of agentic AI is the use of _AI agents_ to perform automated tasks but without human intervention.( While robotic process automation (RPA) and AI agent

In [65]:
print(result['output'])

Agentic AI refers to a type of artificial intelligence focused on autonomous systems capable of making decisions and performing tasks without human intervention.  These systems react independently to conditions to produce results.  It's closely related to agentic automation, used in process management.  Applications include software development, customer support, cybersecurity, and business intelligence.  A key difference from other automation is that agentic AI systems learn and adapt through continuous analysis of data, making decisions independently rather than following fixed rules.


2. Creating ReAct Agent

In [66]:
# import langchain_tavily 
from langchain_tavily import TavilySearch
# create tool
search = TavilySearch(max_results=1)
search_result = search.invoke("Who is Priminister of Pakistan ?")
print(search_result) 

{'query': 'Who is Priminister of Pakistan ?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://x.com/pakpmo?lang=en', 'title': "Prime Minister's Office (@PakPMO) / X", 'content': "Prime Minister's Office (@PakPMO) / X Sign up Prime Minister's Office Prime Minister's Office 3.7M Followers Prime Minister's Office’s posts Prime Minister's Office Prime Minister Muhammad Shehbaz Sharif chairs a consultative meeting on preparation of the Federal Budget 2025-2026. Prime Minister's Office Prime Minister Muhammad Shehbaz Sharif addressing the Nation. Prime Minister's Office Prime Minister Muhammad Shehbaz Sharif addresses the National Assembly session. Prime Minister's Office Prime Minister Muhammad Shehbaz Sharif chairs a meeting of the National Security Committee. Prime Minister's Office Ambassador of Kuwait to Pakistan H.E. Nasser Abdulrahman Jasser called on Prime Minister Muhammad Shehbaz Sharif in Islamabad. Prime Minister's Office Ambassador of UAE 

In [71]:
# make tool of it 
tool = [search]

In [75]:
# import prompt 
from langchain import hub
prompt = hub.pull("hwchase17/react")

In [76]:
# import the agent 
from langchain.agents import create_react_agent
agent = create_react_agent(
    llm=llm,
    tools=tool,
    prompt=prompt
)

# this is a tool_calling_agent

In [77]:
# create the agent_executor 
from langchain.agents import AgentExecutor
agent_executor =AgentExecutor(
    agent=agent,
    tools=tool,
    verbose=True
)

In [80]:
result1 = agent_executor.invoke({"input":"Who is priminister of India and what are there Ongoing Missions ?"})



> Entering new AgentExecutor chain...
Thought: I need to find information about the current Prime Minister of India and their ongoing missions.  A search engine like tavily_search will be helpful for this.

Action: tavily_search

Action Input: "Current Prime Minister of India" AND "ongoing government missions"{'query': 'Current Prime Minister of India" AND "ongoing government missions', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://en.wikipedia.org/wiki/Prime_Minister_of_India', 'title': 'Prime Minister of India - Wikipedia', 'content': 'Modi is the current prime minister of India, serving since 26 May 2014 and ... Prime Minister of India, Government of India. Retrieved 10 April 2018', 'score': 0.572078, 'raw_content': None}], 'response_time': 1.28}Thought: The search result only gives me the name of the Prime Minister. I need to perform another search to find details about his ongoing missions.

Action: tavily_search

Action Input: "Narendra

In [81]:
print(result1['output'])

The current Prime Minister of India is Narendra Modi.  Information regarding his ongoing government missions and initiatives can be found on the official website of the Prime Minister of India (https://www.pmindia.gov.in/en/major-initiatives/).  The "Make in India" campaign is one example of a major initiative mentioned in preliminary searches.  More specific details require accessing and reviewing the content of that website.
